# LG HelloDoctor — B팀 의료 LLM 파인튜닝
> llama3.2-3b + LoRA + Unsloth → Ollama 등록까지

## Step 1 — 라이브러리 설치

In [1]:
!pip install unsloth trl datasets bitsandbytes -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.8/403.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/

## Step 2 — Google Drive 연결

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/LG_HelloDoctor_LLM/checkpoints', exist_ok=True)
print('Drive 연결 완료!')

Mounted at /content/drive
Drive 연결 완료!


## Step 3 — 데이터 로드

In [3]:
from datasets import load_dataset

dataset = load_dataset(
    'json',
    data_files='/content/drive/MyDrive/LG_HelloDoctor_LLM/data/00_all_medical_train.jsonl',
    split='train'
)

print(f'데이터 로드 완료: {len(dataset)}개')
print('샘플 확인:', dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

데이터 로드 완료: 2500개
샘플 확인: {'instruction': '있잖아요, 근데 저기요, 기침이 심해요', 'input': '', 'output': '많이 힘드시겠어요. 내과에 가보시겠어요?'}


## Step 4 — 모델 로드 (4bit 양자화)

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/llama-3.2-3b-instruct',
    max_seq_length=512,
    load_in_4bit=True,
)
print('모델 로드 완료!')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


모델 로드 완료!


## Step 5 — LoRA 설정

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
    bias='none',
    use_gradient_checkpointing=True,
)
print('LoRA 설정 완료!')

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.18 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LoRA 설정 완료!


## Step 6 — 프롬프트 포맷 설정

In [6]:
def format_prompt(example):
    return {
        'text': f"""### 질문:\n{example['instruction']}\n\n### 답변:\n{example['output']}<|end_of_text|>"""
    }

dataset = dataset.map(format_prompt)
print('프롬프트 포맷 완료!')
print('샘플:', dataset[0]['text'])

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

프롬프트 포맷 완료!
샘플: ### 질문:
있잖아요, 근데 저기요, 기침이 심해요

### 답변:
많이 힘드시겠어요. 내과에 가보시겠어요?<|end_of_text|>


## Step 7 — 학습

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=512,
    args=TrainingArguments(
        output_dir='/content/drive/MyDrive/LG_HelloDoctor_LLM/checkpoints',
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=50,
        save_total_limit=3,
        resume_from_checkpoint=True,
    ),
)

trainer.train()
print('학습 완료!')

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2500 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,500 | Num Epochs = 3 | Total steps = 471
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 9,175,040 of 3,221,924,864 (0.28% trained)


Step,Training Loss
10,3.239383
20,1.996426
30,1.610312
40,1.357338
50,1.184113
60,1.077073
70,1.001774
80,0.836300
90,0.830931
100,0.728341


학습 완료!


## Step 8 — 모델 저장

In [8]:
model.save_pretrained('/content/drive/MyDrive/LG_HelloDoctor_LLM/model')
tokenizer.save_pretrained('/content/drive/MyDrive/LG_HelloDoctor_LLM/model')
print('모델 저장 완료!')

모델 저장 완료!


## Step 9 — GGUF 변환

In [9]:
model.save_pretrained_gguf(
    '/content/drive/MyDrive/LG_HelloDoctor_LLM/gguf',
    tokenizer,
    quantization_method='q4_k_m'
)
print('GGUF 변환 완료!')
print('저장 위치: /content/drive/MyDrive/LG_HelloDoctor_LLM/gguf')

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:54<00:00, 27.18s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:06<00:00, 93.18s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/LG_HelloDoctor_LLM/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/LG_HelloDoctor_LLM/gguf_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/LG_HelloDoctor_LLM/gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/drive/MyDrive/LG_HelloDoctor_LLM/gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /content/drive/MyDrive/LG_HelloDoctor_LLM/gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /content/drive/MyDrive/LG_HelloDoctor_LLM/gguf_gguf/Modelfile
GGUF 변환 완료!
저장 위치: /content/drive/MyDrive/LG_HelloDoctor_LLM/gguf


## Step 10 — 의도 분류 테스트

In [10]:
# Step 10 — 추론 테스트 (수정본)

# for_inference 제거하고 generate 설정 수정
def test_model(question):
    inputs = tokenizer(
        f"### 질문:\n{question}\n\n### 답변:\n",
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=100,
        temperature=0.3,
        do_sample=True,
        use_cache=False,   # ← 이게 핵심
        pad_token_id=tokenizer.eos_token_id,
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = result.split("### 답변:")[-1].strip()
    print(f"질문: {question}")
    print(f"답변: {answer}")
    print("-" * 40)

# 테스트
test_model('무릎이 너무 아파요. 어디 가야 해요?')
test_model('집에 가고 싶')
test_model('가슴이 너무 아프고 숨이 안 쉬어져요')
test_model('혈압약이랑 감기약 같이 먹어도 되나요?')

Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

질문: 무릎이 너무 아파요. 어디 가야 해요?
답변: 무릎이 많이 아프시군요. 정형외과에 가보시는 게 좋을 것 같아요.
----------------------------------------


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


질문: 집에 가고 싶
답변: 어디가 불편하신지 말씀해 주시면 도와드릴게요.
----------------------------------------


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


질문: 가슴이 너무 아프고 숨이 안 쉬어져요
답변: 지금 바로 119에 전화해 주세요. 매우 위험한 상황이에요.
----------------------------------------
질문: 혈압약이랑 감기약 같이 먹어도 되나요?
답변: 두 약을 함께 드시면 안 될 수 있어요. 약사 선생님께 한번 여쭤봐 주시겠어요?
----------------------------------------
